# Kriging 5-fold CV — predicted-wet metrics with per-fold std

The chapter-4 table of cross-validation results for the nine
(transform × variogram) configurations reports only the **mean** CRPS and MAE
across the five folds. The supervisor requested the **standard deviation**
across folds as well. This notebook recomputes the per-fold metrics from the
per-record prediction parquets stored on S3 and aggregates them.

**Source data:**
- `s3://thesis-data-ismaktam/kriging/kfold/fold{0..4}_predictions.parquet`
  (5 parquet files, ~2 GB each, $\sim$81 M rows per fold = 9 configs × 9 M test records)

**Pipeline:**
1. Pull each fold's parquet from S3 to a local cache (skip if already cached).
2. Filter to the predicted-wet subset (`predicted_mm >= 0.4` mm) per config.
3. Compute MAE on the median prediction and Gaussian-CRPS on
   $\mathcal{N}(\hat{\mu}, \hat{\sigma}^2)$ with $\hat{\sigma}^2 =$ `predicted_var_mm2`.
4. Aggregate across folds: mean $\pm$ standard deviation.
5. Emit a LaTeX-ready table for [thesis/text/4_kriging.tex](../../thesis/text/4_kriging.tex).

## 0. Imports and paths

In [1]:
import os, time
from pathlib import Path

import numpy as np
import pandas as pd
import boto3
import properscoring as ps

S3_BUCKET = 'thesis-data-ismaktam'
S3_PREFIX = 'kriging/kfold'

CACHE_DIR = Path('/tmp/kriging_predictions')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

PRED_WET_MM = 0.40   # threshold for the 'predicted-wet' subset (matches Hofstra/OK pipeline)
N_FOLDS = 5

s3 = boto3.client('s3')

def s3_pull(fold: int) -> Path:
    """Download fold predictions parquet to local cache if missing; return path."""
    local = CACHE_DIR / f'fold{fold}_predictions.parquet'
    if local.exists() and local.stat().st_size > 1e9:
        return local
    key = f'{S3_PREFIX}/fold{fold}_predictions.parquet'
    t0 = time.time()
    s3.download_file(S3_BUCKET, key, str(local))
    print(f'  fold{fold}: downloaded {local.stat().st_size / 1e9:.2f} GB in {time.time() - t0:.0f} s')
    return local

print(f'S3 source:  s3://{S3_BUCKET}/{S3_PREFIX}/')
print(f'Cache dir:  {CACHE_DIR}')
print(f'PRED_WET_MM = {PRED_WET_MM}')

S3 source:  s3://thesis-data-ismaktam/kriging/kfold/
Cache dir:  /tmp/kriging_predictions
PRED_WET_MM = 0.4


## 1. Per-fold metric computation

For each fold $k$ and each `(transform, variogram_model)` pair:
1. Drop rows with NaN in `predicted_mm` or `predicted_var_mm2`.
2. Restrict to the predicted-wet subset $\hat{Z} \ge 0{.}4$ mm.
3. Compute MAE and Gaussian CRPS.

Gaussian CRPS is used because Ordinary Kriging yields a Gaussian predictive
distribution $\mathcal{N}(\hat{\mu}, \hat{\sigma}^2_K)$. This matches the
fold-0 evaluation in [kriging_fold0_eval.ipynb](kriging_fold0_eval.ipynb).

`ps.crps_gaussian` is the closed-form Gaussian CRPS:
$$\mathrm{CRPS}(\mathcal{N}(\mu, \sigma^2), y) =
  \sigma\Bigl(\,z(2\Phi(z) - 1) + 2\phi(z) - 1/\sqrt{\pi}\Bigr),
  \quad z = (y - \mu)/\sigma.$$

In [2]:
def metrics_one_fold(fold: int) -> pd.DataFrame:
    path = s3_pull(fold)
    t0 = time.time()
    df = pd.read_parquet(path, columns=['transform', 'variogram_model',
                                          'observed_mm', 'predicted_mm',
                                          'predicted_var_mm2'])
    print(f'  fold{fold}: {len(df):,} rows loaded in {time.time() - t0:.1f}s')

    rows = []
    for (t, vm), grp in df.groupby(['transform', 'variogram_model'], observed=True):
        # drop rows with missing predictions / variance
        g = grp.dropna(subset=['predicted_mm', 'predicted_var_mm2'])
        # predicted-wet subset
        pw = g[g['predicted_mm'] >= PRED_WET_MM]
        if len(pw) == 0:
            continue
        obs  = pw['observed_mm'].to_numpy()
        pred = pw['predicted_mm'].to_numpy()
        sd   = np.sqrt(np.clip(pw['predicted_var_mm2'].to_numpy(), 1e-8, None))
        rows.append({
            'fold': fold,
            'transform': t,
            'variogram_model': vm,
            'n_predwet': len(pw),
            'mae':  float(np.mean(np.abs(pred - obs))),
            'crps': float(np.mean(ps.crps_gaussian(obs, mu=pred, sig=sd))),
        })
    return pd.DataFrame(rows)

all_folds = []
print('Computing per-fold predicted-wet metrics ...')
for k in range(N_FOLDS):
    print(f'fold {k}:')
    all_folds.append(metrics_one_fold(k))
perfold = pd.concat(all_folds, ignore_index=True)
print(f'\nResult shape: {perfold.shape}  (expected {N_FOLDS * 9} rows = 5 folds × 9 configs)')
perfold

Computing per-fold predicted-wet metrics ...
fold 0:
  fold0: downloaded 1.94 GB in 360 s
  fold0: 81,593,460 rows loaded in 3.5s
fold 1:
  fold1: 81,386,370 rows loaded in 2.1s
fold 2:
  fold2: 81,386,370 rows loaded in 2.0s
fold 3:
  fold3: 81,386,370 rows loaded in 2.1s
fold 4:
  fold4: 81,386,370 rows loaded in 2.1s

Result shape: (45, 6)  (expected 45 rows = 5 folds × 9 configs)


,fold,transform,variogram_model,n_predwet,mae,crps
0,0,log,exponential,3721640,1.220872,1.002316
1,0,log,gaussian,3721640,1.604487,1.260107
2,0,log,spherical,3721643,1.343528,1.107297
3,0,none,exponential,3721641,1.180463,0.920708
4,0,none,gaussian,3721521,1.633919,1.233283
5,0,none,spherical,3721645,1.362667,1.061924
6,0,normal_score,exponential,3721603,1.211326,0.982474
7,0,normal_score,gaussian,3721631,1.595467,1.229783
8,0,normal_score,spherical,3721627,1.321848,1.071462
9,1,log,exponential,3694333,1.218717,0.995241


## 2. Aggregate: mean ± std across folds

In [4]:
agg = (
    perfold
    .groupby(['transform', 'variogram_model'], observed=True)
    .agg(
        crps_mean = ('crps', 'mean'),
        crps_std  = ('crps', 'std'),
        mae_mean  = ('mae',  'mean'),
        mae_std   = ('mae',  'std'),
        n_total   = ('n_predwet', 'sum'),
    )
    .reset_index()
    .sort_values('crps_mean')
    .reset_index(drop=True)
)
agg

,transform,variogram_model,crps_mean,crps_std,mae_mean,mae_std,n_total
0,none,exponential,0.905493,0.012638,1.162287,0.017132,18524678
1,normal_score,exponential,0.968306,0.010884,1.193654,0.016967,18524640
2,log,exponential,0.988293,0.010748,1.202837,0.017125,18524690
3,none,spherical,1.047957,0.012023,1.346792,0.017095,18524702
4,normal_score,spherical,1.057532,0.010981,1.305509,0.017168,18524676
5,log,spherical,1.093400,0.010922,1.326707,0.017506,18524697
6,normal_score,gaussian,1.214389,0.011320,1.578657,0.015876,18524651
7,none,gaussian,1.219331,0.012293,1.619419,0.017149,18524152
8,log,gaussian,1.244554,0.011472,1.587429,0.016376,18524690


## 4. Save aggregated results

Persisting the aggregated table as CSV and JSON so it can be reused without
re-downloading the 10 GB of fold parquets.

In [6]:
out_dir = Path('../../outputs/kriging/eval')
out_dir.mkdir(parents=True, exist_ok=True)

perfold.to_csv(out_dir / 'cv_predwet_perfold.csv', index=False)
agg.to_csv(out_dir / 'cv_predwet_aggregated.csv', index=False)
print(f'Saved:\n  {out_dir / "cv_predwet_perfold.csv"}\n  {out_dir / "cv_predwet_aggregated.csv"}')

# Compact summary print
print('\n=== predicted-wet (predicted_mm >= 0.4 mm) ===')
print(agg.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

Saved:
  ../../outputs/kriging/eval/cv_predwet_perfold.csv
  ../../outputs/kriging/eval/cv_predwet_aggregated.csv

=== predicted-wet (predicted_mm >= 0.4 mm) ===
   transform variogram_model  crps_mean  crps_std  mae_mean  mae_std  n_total
        none     exponential     0.9055    0.0126    1.1623   0.0171 18524678
normal_score     exponential     0.9683    0.0109    1.1937   0.0170 18524640
         log     exponential     0.9883    0.0107    1.2028   0.0171 18524690
        none       spherical     1.0480    0.0120    1.3468   0.0171 18524702
normal_score       spherical     1.0575    0.0110    1.3055   0.0172 18524676
         log       spherical     1.0934    0.0109    1.3267   0.0175 18524697
normal_score        gaussian     1.2144    0.0113    1.5787   0.0159 18524651
        none        gaussian     1.2193    0.0123    1.6194   0.0171 18524152
         log        gaussian     1.2446    0.0115    1.5874   0.0164 18524690


In [ ]:
def fmt_pm(m, s):
    return f'${m:.3f} \\pm {s:.3f}$'

tx_label = {
    'none':         r'\texttt{none} (ratio) ',
    'normal_score': r'\texttt{normal\_score}',
    'log':          r'\texttt{log}          ',
}

print(r'% --- generated rows for tab:kriging-cv ---')
best_idx = agg['crps_mean'].idxmin()
for i, row in agg.iterrows():
    t  = row['transform']        # bracket access — '.transform' collides with Series.transform method
    vm = row['variogram_model']
    mae_str  = fmt_pm(row.mae_mean,  row.mae_std)
    crps_str = fmt_pm(row.crps_mean, row.crps_std)
    if i == best_idx:
        mae_str  = r'$\mathbf{' + f'{row.mae_mean:.3f}' + r' \pm ' + f'{row.mae_std:.3f}' + r'}$'
        crps_str = r'$\mathbf{' + f'{row.crps_mean:.3f}' + r' \pm ' + f'{row.crps_std:.3f}' + r'}$'
    print(f'    {tx_label[t]} & {vm:<12} & {mae_str} & {crps_str} \\\\')